# 05 · Gold: business-ready tables
| Table | Used by | Grain |
|---|---|---|
| `tender_training` | ML model (MLflow) | one row per contract linked to its tender notice |
| `project_kpis` | dashboard + Data agent | one row per project |
| `supplier_kpis` | dashboard + Data agent | one row per normalised supplier |
| `rag_documents` | Documents agent (search index) | one row per document worth reading |

In [0]:
-- =====================================================================
-- CELL 1: GOLD TENDER_TRAINING -> the model's training data
-- =====================================================================
-- LABEL: tender_days = signing_date - first (non-award) notice date
-- is_valid_label: only 1..1095 days. Negatives / >3 years = linking noise.
-- TARGET LEAKAGE RULES (only use what a planner knows BEFORE the tender):
--   ✅ method, category, review type, sector, estimated size, governance, project age
--   ❌ n_suppliers (known only after award), no_objection_date (after),
--      number of bidders (after bid opening)
-- GOVERNANCE AS-OF JOIN: a score for year Y is published in Y+1,
--   so for a notice in year Y we use the score of year Y-1 (no peeking at the future).
-- MIN_BY(a, b) = the value of a on the row where b is smallest.

CREATE OR REPLACE TABLE procintel.gold.tender_training
COMMENT 'One row per contract linked to its first tender notice. Label = tender_days. Leakage-safe features only.'
AS
WITH first_notice AS (
  SELECT step_activity_id,
         MIN(notice_date)                 AS first_notice_date,
         MIN_BY(notice_type, notice_date) AS first_notice_type,
         COUNT(*)                         AS n_pre_award_notices   -- >1 can mean re-advertising
  FROM procintel.silver.notices
  WHERE step_activity_id <> '' AND notice_type <> 'Contract Award'
  GROUP BY step_activity_id
),
gov AS (   -- PIVOT: one row per year, one column per indicator
  SELECT year,
         MAX(CASE WHEN indicator_code = 'IQ.CPA.PUBS.XQ' THEN score END) AS cpia_public_sector,
         MAX(CASE WHEN indicator_code = 'GOV_WGI_GE.EST' THEN score END) AS wgi_gov_effectiveness,
         MAX(CASE WHEN indicator_code = 'GOV_WGI_RQ.EST' THEN score END) AS wgi_regulatory_quality,
         MAX(CASE WHEN indicator_code = 'GOV_WGI_CC.EST' THEN score END) AS wgi_control_corruption
  FROM procintel.silver.governance_indicators
  GROUP BY year
)
SELECT
  c.contract_id, c.project_id, c.step_activity_id,
  -- features (known at planning time)
  c.procurement_group_code,
  c.procurement_method,
  c.review_type,
  c.major_sector,
  log10(greatest(c.total_amount_usd, 1))                       AS log10_amount_usd,
  n.first_notice_type,
  n.n_pre_award_notices,
  year(n.first_notice_date)                                    AS notice_year,
  month(n.first_notice_date)                                   AS notice_month,
  p.env_category,
  round(months_between(n.first_notice_date, p.approval_date))  AS months_since_approval,
  g.cpia_public_sector, g.wgi_gov_effectiveness, g.wgi_regulatory_quality, g.wgi_control_corruption,
  -- label + its quality flag
  n.first_notice_date, c.signing_date,
  datediff(c.signing_date, n.first_notice_date)                AS tender_days,
  datediff(c.signing_date, n.first_notice_date) BETWEEN 1 AND 1095 AS is_valid_label
FROM procintel.silver.contracts c
JOIN first_notice n               ON c.step_activity_id = n.step_activity_id
LEFT JOIN procintel.silver.projects p ON p.project_id = c.project_id
LEFT JOIN gov g                   ON g.year = year(n.first_notice_date) - 1;

In [0]:
-- =====================================================================
-- CELL 2: GOLD KPI tables -> what the dashboard and the Data agent query
-- =====================================================================
-- NULLIF(x, 0) avoids dividing by zero (returns NULL instead of an error).
-- FILTER (WHERE ...) = apply the aggregate only to rows meeting a condition.

CREATE OR REPLACE TABLE procintel.gold.project_kpis
COMMENT 'One row per project: procurement volume, method mix and tender speed.'
AS
SELECT p.project_id, p.project_name, p.status, p.env_category, p.implementing_agency,
       p.approval_date, p.closing_date, p.total_commitment_usd,
       COUNT(c.contract_id)                                              AS n_contracts,
       SUM(c.total_amount_usd)                                           AS contracted_usd,
       ROUND(100 * COUNT_IF(c.procurement_method = 'Direct Selection')
             / NULLIF(COUNT(c.contract_id), 0), 1)                       AS pct_direct_selection,
       median(t.tender_days) FILTER (WHERE t.is_valid_label)             AS median_tender_days
FROM procintel.silver.projects p
LEFT JOIN procintel.silver.contracts c       ON c.project_id  = p.project_id
LEFT JOIN procintel.gold.tender_training t   ON t.contract_id = c.contract_id
GROUP BY ALL;   -- GROUP BY ALL = group by every non-aggregated column

CREATE OR REPLACE TABLE procintel.gold.supplier_kpis
COMMENT 'One row per normalised supplier. supplier_name may be an individual -> PII.'
AS
SELECT supplier_key,
       MIN(supplier_name)            AS example_name,
       COUNT(DISTINCT contract_id)   AS n_contracts,
       COUNT(DISTINCT project_id)    AS n_projects,
       SUM(supplier_amount_usd)      AS total_awarded_usd
FROM procintel.silver.contract_suppliers
WHERE supplier_key <> ''
GROUP BY supplier_key;

ALTER TABLE procintel.gold.supplier_kpis ALTER COLUMN example_name SET TAGS ('pii' = 'possible');

In [0]:
-- =====================================================================
-- CELL 3: GOLD RAG_DOCUMENTS -> the documents worth reading
-- =====================================================================
-- RAG = Retrieval-Augmented Generation: the chatbot SEARCHES these documents
-- and answers from what it finds (with citations) instead of from memory.
-- We keep only the 4 families that explain WHY things went well or badly,
-- and only documents that have a plain-text version (txturl).

CREATE OR REPLACE TABLE procintel.gold.rag_documents
COMMENT 'Documents selected for the search index: ICR, IEG reviews, PAD, ISR with a text URL.'
AS
SELECT d.*, p.project_name
FROM procintel.silver.documents d
LEFT JOIN procintel.silver.projects p USING (project_id)
WHERE d.doc_family IN ('ICR', 'IEG_ICRR', 'IEG_PPAR', 'PAD', 'ISR')
  AND d.txturl IS NOT NULL;

In [0]:
-- =====================================================================
-- CELL 4: GOLD REPORT -> how big and how clean is each Gold table?
-- =====================================================================
SELECT 'tender_training: all linked'   AS metric, CAST(COUNT(*) AS STRING) AS value FROM procintel.gold.tender_training
UNION ALL SELECT 'tender_training: valid labels', CAST(COUNT_IF(is_valid_label) AS STRING) FROM procintel.gold.tender_training
UNION ALL SELECT 'tender_training: removed as noise', CAST(COUNT_IF(NOT is_valid_label OR is_valid_label IS NULL) AS STRING) FROM procintel.gold.tender_training
UNION ALL SELECT 'valid: median tender days', CAST(median(tender_days) FILTER (WHERE is_valid_label) AS STRING) FROM procintel.gold.tender_training
UNION ALL SELECT 'valid: % with governance scores', CAST(ROUND(100*COUNT_IF(is_valid_label AND cpia_public_sector IS NOT NULL)/COUNT_IF(is_valid_label),1) AS STRING) FROM procintel.gold.tender_training
UNION ALL SELECT 'project_kpis rows', CAST(COUNT(*) AS STRING) FROM procintel.gold.project_kpis
UNION ALL SELECT 'supplier_kpis rows (after name normalisation)', CAST(COUNT(*) AS STRING) FROM procintel.gold.supplier_kpis
UNION ALL SELECT 'supplier raw names (before)', CAST(COUNT(DISTINCT supplier_name) AS STRING) FROM procintel.silver.contract_suppliers
UNION ALL SELECT 'rag_documents', CAST(COUNT(*) AS STRING) FROM procintel.gold.rag_documents;